**Assignment Overview**

In this assignment, you will explore two fundamental aspects of modern NLP systems: fine-tuning large language models and understanding the attention mechanism that powers them.

**Learning Goals**

By the end of this assignment, you should be able to:

* Understand when and why to fine-tune language models
* Apply QLoRA for efficient adaptation of large models
* Analyze model outputs and limitations
* Explain and implement the attention mechanism
* Interpret attention patterns and their behavior

**Important Note**

* Do not modify or delete the task structure.
* Complete each task with:
   - Clean, well-organized code
   - Relevant visualizations
   - Clear insights and explanations
* Make sure to answer all required questions.

**Submission requirements:**
* Submit a **fully executed notebook** (all cells must run and outputs should be visible).
* There is no need to attach the training and test datasets as files, but you must present them as DataFrame tables within the notebook.

## Install Required Packages and Dataset

In [2]:
%pip install -q -U python-dotenv bitsandbytes peft trl fastapi uvicorn openai datasets tqdm torchao ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    get_peft_model,
    PeftModel,
)
from trl import SFTTrainer
import torch
from dotenv import load_dotenv
import json
from datasets import Dataset, load_dataset
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
logging.set_verbosity(logging.CRITICAL)

load_dotenv()

W0515 17:28:22.757000 26526 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


True

## **Part 1 — QLoRA Fine-Tuning for Level-Adaptive Question Answering (75 points)**

In this part, you will fine-tune pretrained language models to answer questions at different explanation levels:

* Child — simple and intuitive explanation
* Student — clear educational explanation with moderate technical detail
* Expert — precise, technical, and domain-specific explanation

You will use a subset of the Databricks Dolly 15K dataset.

In [4]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:5000]")

In [5]:
def is_good_question(text):
    return any(q in text.lower() for q in ["why", "how", "what is", "explain"])


filtered = [
    ex
    for ex in dataset
    if ex["category"] == "open_qa" and is_good_question(ex["instruction"])
]

In [6]:
len(filtered), filtered[0]

(580,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

### **Task 1.1 — Prepare a Custom Dataset (15 points)**

Create your own instruction-tuning dataset based on a subset of databricks-dolly-15k.

For each selected question-answer pair, create versions of the answer adapted to the requested expertise level.

Example format:

    Question: What is gradient descent?
    Expertise level: child
    Answer: Gradient descent is like walking downhill step by step until you reach the lowest point.

You should prepare:

1. Training set for fine-tuning -

   Use the filtered Dolly dataset as your source of questions.

   For each question, create examples in the example format per level.

2. Test set for evaluation -
    
   Create a small test set of 20 question-answer examples.

   The test set must:

  * include examples from all three expertise levels
  * be separate from the training set
  * be manually reviewed by you for quality
  * include answers that are appropriate for the requested expertise level

  Recommendation:

  1. Save the generated dataset as a '.jsonl' file so it can be loaded and reused later.

  2. Decide on the training format according to the model architecture:
    
      * For causal language models, such as `HuggingFaceTB/SmolLM2-360M-Instruct`, use one full text field:
         ```text
            ### Question:
            ...

            ### Expertise level:
            child / student / expert

            ### Answer:
            ...
         ```

      * For seq2seq models, such as google/flan-t5-small, separate the input and target:
         * input: Question + expertise level
         * target: Answer
   
 3. Before generating the full dataset, test the pipeline on a small batch of examples to verify that:

  * the LLM returns valid JSON,
  * each question receives three expertise-level answers,
  * the saved .jsonl file can be loaded correctly,
  * the format matches the training code.

In [9]:
import os
import time
from openai import OpenAI
from tqdm import tqdm

# constants

COLAB = False

if COLAB:
    from google.colab import userdata

    NEBIUS_API_KEY = userdata.get("NEBIUS_API_KEY")
else:
    load_dotenv()
    NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")


# ── Configuration ──────────────────────────────────────────────────────────────
DEBUG = False  # True = process only 3 questions for testing; False = all ~580
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY", "YOUR-NEBIUS-KEY-HERE")
MODEL = "meta-llama/Llama-3.3-70B-Instruct"
OUTPUT_DIR = "."

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/", api_key=NEBIUS_API_KEY
)

LEVELS = ["child", "student", "expert"]


print("Client ready ✓")

Client ready ✓


In [10]:
# ── Step 1: Generate expertise-level answers via OpenAI ────────────────────────


def generate_level_answers(
    question: str, original_answer: str, max_retries: int = 3
) -> dict:
    """Call OpenAI to produce child / student / expert answers for a question."""
    prompt = f"""You are given a question and a reference answer.
Rewrite the answer at three expertise levels.

Question: {question}
Reference answer: {original_answer}

Return ONLY a JSON object with exactly three keys: "child", "student", "expert".
- "child": simple, intuitive explanation a 7-year-old would understand.
- "student": clear educational explanation with moderate technical detail.
- "expert": precise, technical, domain-specific explanation.

Each value must be a single string (1-3 sentences). No markdown, no extra keys."""

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0.7,
                messages=[
                    {
                        "role": "system",
                        "content": "You return only valid JSON. No markdown fences, no extra text.",
                    },
                    {"role": "user", "content": prompt},
                ],
            )
            raw = resp.choices[0].message.content.strip()

            if raw.startswith("```"):
                raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

            data = json.loads(raw)

            if all(k in data for k in LEVELS):
                return data

            raise ValueError(f"Missing keys: {set(LEVELS) - set(data.keys())}")
        except Exception as e:
            if attempt == max_retries - 1:
                print(
                    f"  ✗ Failed after {max_retries} retries for: {question[:60]}… — {e}"
                )
                return None

            time.sleep(2**attempt)


# ── Step 2: Formatting helpers ─────────────────────────────────────────────────


def format_causal(question: str, level: str, answer: str) -> str:
    """SmolLM2 causal-LM format: single text field."""
    return f"### Question:\n{question}\n\n### Expertise level:\n{level}\n\n### Answer:\n{answer}"


def format_seq2seq_input(question: str, level: str) -> str:
    """flan-t5 seq2seq format: input field."""
    return f"Question: {question}\nExpertise level: {level}"


def build_rows(question: str, answers: dict) -> list[dict]:
    """Expand one question + 3-level answers into 3 dataset rows."""
    rows = []
    for level in LEVELS:
        answer = answers[level]
        rows.append(
            {
                "question": question,
                "level": level,
                "answer": answer,
                "text": format_causal(question, level, answer),
                "input": format_seq2seq_input(question, level),
                "target": answer,
            }
        )
    return rows


# ── Step 3: Batch processing ──────────────────────────────────────────────────

source = filtered[:3] if DEBUG else filtered
all_rows = []

print(f"Processing {len(source)} questions (DEBUG={DEBUG})…\n")

for ex in tqdm(source, desc="Generating answers"):
    answers = generate_level_answers(ex["instruction"], ex["response"])
    if answers is not None:
        all_rows.extend(build_rows(ex["instruction"], answers))
    time.sleep(0.3)

print(
    f"\nGenerated {len(all_rows)} total examples from {len(all_rows) // 3} questions."
)


# ── Step 4: Train / test split ────────────────────────────────────────────────
# Reserve the last 7 questions (= 21 examples, ≥ 20 required) for the test set.
# In DEBUG mode, reserve 1 question for test so both splits are non-empty.

n_questions = len(all_rows) // len(LEVELS)
n_test_questions = 1 if DEBUG else 7
n_test_rows = n_test_questions * len(LEVELS)

train_rows = all_rows[:-n_test_rows]
test_rows = all_rows[-n_test_rows:]

print(
    f"Train: {len(train_rows)} examples ({len(train_rows) // 3} Qs)  |  Test: {len(test_rows)} examples ({len(test_rows) // 3} Qs)"
)


# ── Step 5: Save to JSONL ────────────────────────────────────────────────────

train_path = os.path.join(OUTPUT_DIR, "train.jsonl")
test_path = os.path.join(OUTPUT_DIR, "test.jsonl")

for path, rows in [(train_path, train_rows), (test_path, test_rows)]:
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(
    f"Saved {train_path} ({len(train_rows)} rows) and {test_path} ({len(test_rows)} rows)"
)


# ── Step 6: Verify reload ────────────────────────────────────────────────────

if len(all_rows) == 0:
    print("⚠ No examples generated — check your API key and try again.")
else:
    train_ds = load_dataset("json", data_files=train_path, split="train")
    test_ds = load_dataset("json", data_files=test_path, split="train")
    print(f"Reloaded OK — train: {len(train_ds)}, test: {len(test_ds)}")


# ── Step 7: Display as DataFrames ────────────────────────────────────────────

train_df = pd.DataFrame(train_rows)
test_df = pd.DataFrame(test_rows)

print("\n─── Training set (first 6 rows) ───")
display(train_df.head(6))

print("\n─── Test set (all rows) ───")
display(test_df)

Processing 580 questions (DEBUG=False)…



Generating answers: 100%|██████████| 580/580 [1:00:57<00:00,  6.31s/it]



Generated 1740 total examples from 580 questions.
Train: 1719 examples (573 Qs)  |  Test: 21 examples (7 Qs)
Saved ./train.jsonl (1719 rows) and ./test.jsonl (21 rows)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Reloaded OK — train: 1719, test: 21

─── Training set (first 6 rows) ───


,question,level,answer,text,input,target
0,Why can camels survive for long without water?,child,Camels have a special hump on their back that ...,### Question:\nWhy can camels survive for long...,Question: Why can camels survive for long with...,Camels have a special hump on their back that ...
1,Why can camels survive for long without water?,student,Camels can survive for long periods without wa...,### Question:\nWhy can camels survive for long...,Question: Why can camels survive for long with...,Camels can survive for long periods without wa...
2,Why can camels survive for long without water?,expert,Camels' ability to survive without water for e...,### Question:\nWhy can camels survive for long...,Question: Why can camels survive for long with...,Camels' ability to survive without water for e...
3,What is a polygon?,child,A polygon is a shape with straight lines and c...,### Question:\nWhat is a polygon?\n\n### Exper...,Question: What is a polygon?\nExpertise level:...,A polygon is a shape with straight lines and c...
4,What is a polygon?,student,A polygon is a two-dimensional shape with a fi...,### Question:\nWhat is a polygon?\n\n### Exper...,Question: What is a polygon?\nExpertise level:...,A polygon is a two-dimensional shape with a fi...
5,What is a polygon?,expert,"A polygon is a planar, closed curve composed o...",### Question:\nWhat is a polygon?\n\n### Exper...,Question: What is a polygon?\nExpertise level:...,"A polygon is a planar, closed curve composed o..."



─── Test set (all rows) ───


,question,level,answer,text,input,target
0,What is the difference between a simile and a ...,child,A simile is when we say something is like some...,### Question:\nWhat is the difference between ...,Question: What is the difference between a sim...,A simile is when we say something is like some...
1,What is the difference between a simile and a ...,student,Similes and metaphors are literary devices use...,### Question:\nWhat is the difference between ...,Question: What is the difference between a sim...,Similes and metaphors are literary devices use...
2,What is the difference between a simile and a ...,expert,"Similes employ explicit comparative markers, s...",### Question:\nWhat is the difference between ...,Question: What is the difference between a sim...,"Similes employ explicit comparative markers, s..."
3,"If I have a dollar, two nickels and a quarter,...",child,$1 + $0.10 + $0.10 + $0.25 = $1.35,"### Question:\nIf I have a dollar, two nickels...","Question: If I have a dollar, two nickels and ...",$1 + $0.10 + $0.10 + $0.25 = $1.35
4,"If I have a dollar, two nickels and a quarter,...",student,"To find the total, add the values: $1 (dollar)...","### Question:\nIf I have a dollar, two nickels...","Question: If I have a dollar, two nickels and ...","To find the total, add the values: $1 (dollar)..."
5,"If I have a dollar, two nickels and a quarter,...",expert,$1 + 2 * $0.05 + $0.25 = $1.35,"### Question:\nIf I have a dollar, two nickels...","Question: If I have a dollar, two nickels and ...",$1 + 2 * $0.05 + $0.25 = $1.35
6,What is the Genomie Aggregation Databaise (gen...,child,The Genome Aggregation Database is a big libra...,### Question:\nWhat is the Genomie Aggregation...,Question: What is the Genomie Aggregation Data...,The Genome Aggregation Database is a big libra...
7,What is the Genomie Aggregation Databaise (gen...,student,"The Genome Aggregation Database, or genomAD, i...",### Question:\nWhat is the Genomie Aggregation...,Question: What is the Genomie Aggregation Data...,"The Genome Aggregation Database, or genomAD, i..."
8,What is the Genomie Aggregation Databaise (gen...,expert,The Genome Aggregation Database (genomAD) is a...,### Question:\nWhat is the Genomie Aggregation...,Question: What is the Genomie Aggregation Data...,The Genome Aggregation Database (genomAD) is a...
9,How do I find the best interior decorator and ...,child,"To find the best interior decorator, ask frien...",### Question:\nHow do I find the best interior...,Question: How do I find the best interior deco...,"To find the best interior decorator, ask frien..."


### **Task 1.2 — Fine-Tune Models with QLoRA (10 points)**

Fine-tune the models using QLoRA, meaning:

1. Load the base model in 4-bit quantization
2. Add LoRA adapters
3. Train only the adapter parameters

You should repeat the fine-tuning process for both models:

1. HuggingFaceTB/SmolLM2-360M-Instruct
2. google/flan-t5-small

Pay attention - each model requires the dataset to be formatted differently.

In [8]:
# Task 1.2 — QLoRA fine-tuning (SmolLM2 causal LM + flan-t5 seq2seq)
# Requires a CUDA GPU (enable T4 in Google Colab). Upload train.jsonl to the runtime if needed.

import gc
import os

from trl import SFTTrainer, SFTConfig

if not torch.cuda.is_available():
    raise RuntimeError(
        "QLoRA needs a CUDA GPU. In Colab: Runtime → Change runtime type → GPU."
    )

try:
    _notebook_dir = _dh[0]  # Jupyter / Colab: directory containing the notebook
except NameError:
    _notebook_dir = os.getcwd()

_train_candidates = [
    os.path.join(_notebook_dir, "train.jsonl"),
    os.path.join(os.getcwd(), "train.jsonl"),
    "/content/train.jsonl",
    "train.jsonl",
]
train_jsonl = None

for p in _train_candidates:
    if os.path.isfile(p):
        train_jsonl = p
        break

if train_jsonl is None:
    train_jsonl = "train.jsonl"  # Hugging Face datasets will error clearly if missing

print(f"Loading training data from: {train_jsonl}")
train_ds = load_dataset("json", data_files=train_jsonl, split="train")
print(f"Train examples: {len(train_ds)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ── 1) SmolLM2 — causal LM + SFTTrainer ───────────────────────────────────────

smollm_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
smollm_tok = AutoTokenizer.from_pretrained(smollm_id, trust_remote_code=True)

if smollm_tok.pad_token is None:
    smollm_tok.pad_token = smollm_tok.eos_token

smollm_model = AutoModelForCausalLM.from_pretrained(
    smollm_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
smollm_model = prepare_model_for_kbit_training(smollm_model)

smollm_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
)
smollm_model = get_peft_model(smollm_model, smollm_lora)
smollm_model.print_trainable_parameters()

smollm_args = SFTConfig(
    output_dir="./smollm2-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

smollm_trainer = SFTTrainer(
    model=smollm_model,
    args=smollm_args,
    train_dataset=train_ds,
    processing_class=smollm_tok,
)

print("Training SmolLM2 + QLoRA…")
smollm_trainer.train()
smollm_trainer.save_model("./smollm2-qlora")
print("Saved SmolLM2 adapter to ./smollm2-qlora")

del smollm_trainer, smollm_model
gc.collect()
torch.cuda.empty_cache()

# ── 2) flan-t5-small — seq2seq + Seq2SeqTrainer ───────────────────────────────

t5_id = "google/flan-t5-small"
t5_tok = AutoTokenizer.from_pretrained(t5_id)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    t5_id,
    quantization_config=bnb_config,
    device_map="auto",
)
t5_model = prepare_model_for_kbit_training(t5_model)

t5_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q", "v"],
)
t5_model = get_peft_model(t5_model, t5_lora)
t5_model.print_trainable_parameters()


def tokenize_t5(batch):
    model_inputs = t5_tok(
        batch["input"], truncation=True, max_length=512, padding=False
    )
    labels = t5_tok(
        text_target=batch["target"], truncation=True, max_length=256, padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


t5_train = train_ds.map(
    tokenize_t5,
    batched=True,
    remove_columns=train_ds.column_names,
)

t5_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=t5_tok, model=t5_model, padding=True)

t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_args,
    train_dataset=t5_train,
    data_collator=data_collator,
    processing_class=t5_tok,
)

print("Training flan-t5-small + QLoRA…")
t5_trainer.train()
t5_trainer.save_model("./flan-t5-qlora")
print("Saved flan-t5 adapter to ./flan-t5-qlora")

RuntimeError: QLoRA needs a CUDA GPU. In Colab: Runtime → Change runtime type → GPU.

### **Task 1.3 — Evaluation (20 points)**

Evaluate both fine-tuned models on your 20-example test set.

For each model, compare:

* outputs before fine-tuning
* outputs after fine-tuning
* whether the answer matches the requested expertise level
* whether the answer is clear and relevant
* whether the answer stays faithful to the question

Evaluate using this table:

|Question|Level|Base Output|Fine-tuned Output|Expected Answer|Better after FT?|Notes|
|-------|-------|-------|-------|-------|-------|-------|

### **Task 1.4 — Model Comparison and Discussion (15 points)**

Compare the performance of the two models:

* Which model adapted better to the expertise levels?
* Which model produced clearer answers?
* Which model followed the requested format better?
* Did one model hallucinate more than the other?

Explain any differences you observe.

In your discussion, consider that:

* SmolLM2-360M-Instruct is a decoder-only instruction model
* flan-t5-small is an encoder-decoder instruction model
* Different architectures may behave differently on instruction-following and text generation tasks

### **Task 1.5 - Conceptual Questions (15 points)**

Answer the following questions:

1. What changed after fine-tuning?
   
   Discuss whether the model became better at adapting its answer to the requested expertise level.
2. Why is QLoRA more memory efficient?
   
   Explain the role of 4-bit quantization and LoRA adapters.
3. What happens if you increase the LoRA rank?
   
   Discuss the tradeoff between model capacity, memory usage, and overfitting risk.
4. Why use LoRA / QLoRA instead of prompt engineering?

      Discuss:

      * In what cases prompt engineering is sufficient
      * When fine-tuning becomes necessary
      * What advantages QLoRA provides over prompting
      * What are the trade-offs (cost, flexibility, control)

## **Part 2 — Understanding Attention (25 points)**

Goal:

Build intuition for how attention works.

Task:

You will implement a simple attention mechanism from scratch (PyTorch) and visualize its behavior.

### **Task 2.1 Implement Scaled Dot-Product Attention (5 points)**

Given:

* Query (Q)
* Key (K)
* Value (V)

Compute:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Scaled Dot-Product Attention.

    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: (batch, heads, seq_len_q, seq_len_k) or None

    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """


    #  Get dimension for scaling
    d_k = Q.size(-1)


    # Compute attention scores

    scores = # TODO: compute dot-product between Q and K^T


    # Scale the scores

    # TODO: divide scores by sqrt(d_k)


    #  Apply mask (if given)

    if mask is not None:
        # TODO: mask out invalid positions (set to -inf)
        pass


    # Softmax to get attention

    attention_weights = # TODO: apply softmax over last dimension


    # Compute weighted sum

    output = #TODO: multiply attention_weights with V

    return output, attention_weights

### **Task 2.2 Visualize Attention (5 points)**

Plot attention weights as a heatmap for the given sentence

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import math



sentence = "the cat sat on the mat"


tokens = # TODO: split sentence into tokens

seq_len = # TODO: compute sequence length



# Create dummy embeddings

d_model = 16


embeddings = # TODO: initialize random embeddings of shape (seq_len, d_model)

# Treat embeddings as Q, K, V
Q = embeddings
K = embeddings
V = embeddings



# Compute Scaled Dot-Product Attention

attention_weights = # TODO: compute attention weights (use function from above)



# Plot attention heatmap

plt.figure(figsize=(8, 6))

# TODO: plot heatmap using seaborn
# - use attention_weights
# - set xticklabels and yticklabels to tokens
# - choose a colormap (e.g., "viridis")

plt.title("Attention Weights Heatmap")

# TODO: label axes
# plt.xlabel(...)
# plt.ylabel(...)

# TODO: rotate ticks if needed

plt.show()

### **Task 2.3 Experiments (15 points)**

**Experiment 1 — Change One Word:**

1. Use a simple sentence, for example:
    the cat sat on the mat
2. Compute and plot the attention weights as a heatmap.
3. Change one word in the sentence, for example:
    the dog sat on the mat
4. Recompute and plot the attention heatmap.
5. Compare the two heatmaps and explain whether the attention pattern changed.

**Experiment 2 — Compare Different Attention Heads:**

Repeat the attention visualization using at least two different attention heads.

For each head, create separate projection matrices:

$$W_Q, W_K, W_V$$

Use them to compute:
$$Q=XW_Q, K=XW_K, V=XW_V$$

Then compute and plot the attention weights for each head.

**Questions**

Answer briefly:

1. Did changing one word affect the attention weights? Why or why not?
2. Do different attention heads focus on different tokens?
3. Why might multi-head attention be useful in Transformer models?